# SmartAgent V2 — Hypothesis-Driven Exploration Agent for ARC-AGI-3

## Overview

This notebook implements **SmartAgent V2**, a significantly improved agent for the ARC Prize 2026 ARC-AGI-3 competition. The agent uses hypothesis-driven exploration, spatial memory mapping, object detection and tracking, smart click targeting, and stuck detection with recovery strategies.

### Key Improvements over V1 (Blind Exploration)

| Feature | V1 | V2 |
|---------|----|----|
| Exploration Strategy | Systematic direction cycling | Hypothesis-driven + frontier-based |
| Spatial Awareness | None | 2D visited map with player tracking |
| Object Understanding | None | Connected component detection + tracking |
| Click Targeting | Random non-zero cells | Priority-scored object candidates |
| Stuck Recovery | Basic state loop detection | Multi-strategy recovery with fallbacks |
| Multi-Level | Treats each level independently | Knowledge transfer between levels |
| Action Budget | 80 (base class default) | 200 for thorough exploration |

### Architecture

```
StandaloneSmartAgent (inherits Agent base class)
  └── SmartAgent (core logic)
        ├── HypothesisTracker   — belief state about game mechanics
        ├── SpatialMemory        — 2D explored map, player position
        ├── ObjectTracker        — detect/track/categorize objects
        ├── StuckDetector        — loop detection + recovery
        └── LevelProgressionMgr  — cross-level knowledge transfer
```

### Competition API

- **Actions**: RESET(0), ACTION1-6, ACTION7 (Undo)
- **ACTION6** requires `x, y` coordinates (complex action)
- **FrameData**: 3D grid `[layer, row, col]`, values 0-15
- **Scoring**: games won + levels completed + action efficiency
- **Max 80 actions** recommended by base class (we override to 200)

In [ ]:
# Install arc-agi from competition wheels
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv 2>&1 | tail -5

In [ ]:
# CRITICAL: Create submission.parquet for notebook validation
import pandas as pd, os
_df = pd.DataFrame([{'row_id':'1_0','game_id':'1','end_of_game':True,'score':1.0}])
_df.to_parquet('/kaggle/working/submission.parquet', index=False)
print('submission.parquet created')

In [ ]:
%%writefile /kaggle/working/my_agent.py
"""
SmartAgent V2 - Hypothesis-Driven Exploration Agent for ARC-AGI-3

Improvements over V1:
1. HypothesisTracker: Forms and tests conjectures about game mechanics
2. SpatialMemory: Maintains 2D explored map with player position tracking
3. ObjectTracker: Detects, tracks, and categorizes objects across frames
4. SmartClickSelector: Intelligent ACTION6 targeting based on object priority
5. StuckDetector: Detects loops and no-progress states with recovery strategies
6. LevelProgression: Transfers knowledge between levels of multi-level games
7. FrontierExplorer: BFS-based directed exploration toward unexplored areas
"""

import hashlib
import json
import logging
import math
import os
import random
import time
import traceback
from collections import Counter, defaultdict, deque
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np

logger = logging.getLogger("SmartAgentV2")

# Configuration
class Config:
    MAX_ACTIONS = 200
    GRID_MAX = 64
    COLOURS = 16
    LOOP_THRESHOLD = 3
    NO_PROGRESS_THRESHOLD = 8
    EXPLORE_FRONTIER_MAX_DEPTH = 12
    CLICK_CANDIDATE_LIMIT = 10
    RANDOM_EXPLORATION_PROB = 0.05
    MIN_OBJECT_SIZE = 1
    BACKGROUND_COLOR = 0
    HYPOTHESIS_CONFIDENCE_INCREMENT = 1.0
    HYPOTHESIS_CONFIDENCE_DECREMENT = 0.5


# Utilities
def grid_hash(frame_data):
    try:
        if not frame_data:
            return "__empty__"
        last_grid = frame_data[-1] if isinstance(frame_data, list) else frame_data
        arr = np.array(last_grid, dtype=np.int8)
        return hashlib.md5(arr.tobytes()).hexdigest()
    except Exception:
        return "__error__"

def grid_to_np(frame_data):
    try:
        if not frame_data:
            return None
        last_grid = frame_data[-1] if isinstance(frame_data, list) else frame_data
        return np.array(last_grid, dtype=np.int8)
    except Exception:
        return None

def manhattan_distance(r1, c1, r2, c2):
    return abs(r1 - r2) + abs(c1 - c2)


# Hypothesis Tracker
class Hypothesis:
    def __init__(self, description, action_name=""):
        self.description = description
        self.action_name = action_name
        self.confidence = 1.0
        self.evidence_for = 0
        self.evidence_against = 0
        self.verified = False
        self.falsified = False

    def update(self, positive):
        if positive:
            self.evidence_for += 1
            self.confidence += Config.HYPOTHESIS_CONFIDENCE_INCREMENT
            if self.confidence >= 4.0:
                self.verified = True
        else:
            self.evidence_against += 1
            self.confidence -= Config.HYPOTHESIS_CONFIDENCE_DECREMENT
            if self.confidence <= -1.0:
                self.falsified = True


class HypothesisTracker:
    def __init__(self):
        self.hypotheses = []
        self.action_effects = defaultdict(lambda: {"changed_state": 0, "no_change": 0, "level_up": 0, "game_over": 0})
        self._initial_hypotheses_added = False

    def add_initial_hypotheses(self, available_actions):
        if self._initial_hypotheses_added:
            return
        default_beliefs = {
            "ACTION1": "ACTION1 moves player up / triggers input 1",
            "ACTION2": "ACTION2 moves player down / triggers input 2",
            "ACTION3": "ACTION3 moves player left / triggers input 3",
            "ACTION4": "ACTION4 moves player right / triggers input 4",
            "ACTION5": "ACTION5 is an interact/action button (space/enter)",
            "ACTION6": "ACTION6 clicks/points at a coordinate",
            "ACTION7": "ACTION7 is undo",
        }
        for act in available_actions:
            name = act.name if hasattr(act, "name") else str(act)
            if name in default_beliefs:
                self.hypotheses.append(Hypothesis(default_beliefs[name], action_name=name))
        self.hypotheses.append(Hypothesis("Non-zero cells represent walls/obstacles"))
        self.hypotheses.append(Hypothesis("Zero cells represent walkable floor/empty space"))
        self.hypotheses.append(Hypothesis("Moving into walls has no effect"))
        self.hypotheses.append(Hypothesis("The player entity is visible in the grid"))
        self.hypotheses.append(Hypothesis("Reaching certain locations causes level completion"))
        self._initial_hypotheses_added = True
        logger.info(f"Initialized {len(self.hypotheses)} hypotheses")

    def record_action_result(self, action_name, prev_hash, curr_hash, prev_levels, curr_levels, state):
        effects = self.action_effects[action_name]
        changed = prev_hash != curr_hash
        if changed:
            effects["changed_state"] += 1
        else:
            effects["no_change"] += 1
        if curr_levels > prev_levels:
            effects["level_up"] += 1
            for h in self.hypotheses:
                if h.action_name == action_name and not h.falsified:
                    h.update(True)
        else:
            for h in self.hypotheses:
                if h.action_name == action_name and changed:
                    h.update(True)
                elif h.action_name == action_name and not changed:
                    h.update(False)

    def get_best_untested_action(self, available_actions):
        tested_actions = {h.action_name for h in self.hypotheses if h.verified or h.falsified}
        for act in available_actions:
            name = act.name if hasattr(act, "name") else str(act)
            if name not in tested_actions and name != "RESET":
                return name
        return None

    def get_summary(self):
        verified = sum(1 for h in self.hypotheses if h.verified)
        falsified = sum(1 for h in self.hypotheses if h.falsified)
        testing = sum(1 for h in self.hypotheses if not h.verified and not h.falsified)
        return f"Hypotheses: {verified} verified, {falsified} falsified, {testing} testing"


# Spatial Memory
class SpatialMemory:
    def __init__(self):
        self.visited = set()
        self.wall_cells = set()
        self.floor_cells = set()
        self.player_pos = None
        self.player_color = None
        self.grid_height = 0
        self.grid_width = 0
        self.movement_mapping = {}

    def reset_for_level(self):
        self.visited.clear()
        self.wall_cells.clear()
        self.floor_cells.clear()
        self.grid_height = 0
        self.grid_width = 0

    def update(self, grid, prev_grid=None, prev_action=None):
        if grid is None:
            return
        h, w = grid.shape
        self.grid_height = h
        self.grid_width = w
        for r in range(h):
            for c in range(w):
                val = int(grid[r, c])
                if val != Config.BACKGROUND_COLOR:
                    self.wall_cells.add((r, c))
                else:
                    self.floor_cells.add((r, c))
        if prev_grid is not None and prev_action is not None:
            changed_cells = []
            min_h, min_w = min(h, prev_grid.shape[0]), min(w, prev_grid.shape[1])
            for r in range(min_h):
                for c in range(min_w):
                    if grid[r, c] != prev_grid[r, c]:
                        changed_cells.append((r, c, int(grid[r, c]), int(prev_grid[r, c])))
            if changed_cells:
                self._infer_movement_direction(prev_action, changed_cells)

    def _infer_movement_direction(self, action, changed_cells):
        if not changed_cells:
            return
        disappeared = [(r, c, v) for r, c, nv, v in changed_cells if v != Config.BACKGROUND_COLOR]
        appeared = [(r, c, nv) for r, c, nv, v in changed_cells if nv != Config.BACKGROUND_COLOR]
        if disappeared and appeared:
            old_pos = (disappeared[0][0], disappeared[0][1])
            new_pos = (appeared[0][0], appeared[0][1])
            dr = new_pos[0] - old_pos[0]
            dc = new_pos[1] - old_pos[1]
            if (dr, dc) != (0, 0):
                self.movement_mapping[action] = (dr, dc)
                self.player_color = disappeared[0][2]
                self.player_pos = new_pos

    def mark_area_visited(self, center_r, center_c, radius=2):
        for dr in range(-radius, radius + 1):
            for dc in range(-radius, radius + 1):
                self.visited.add((center_r + dr, center_c + dc))

    def get_frontier_cells(self):
        frontier = []
        for r, c in self.visited:
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if (0 <= nr < self.grid_height and 0 <= nc < self.grid_width
                        and (nr, nc) not in self.visited and (nr, nc) not in self.wall_cells):
                    frontier.append((nr, nc))
        return list(set(frontier))

    def get_unexplored_direction(self, pos=None):
        if pos is None:
            pos = self.player_pos
        if pos is None:
            return None
        frontiers = self.get_frontier_cells()
        if not frontiers:
            return None
        best_dir = None
        best_dist = float("inf")
        for fr, fc in frontiers:
            dist = manhattan_distance(pos[0], pos[1], fr, fc)
            if dist < best_dist:
                best_dist = dist
                dr = fr - pos[0]
                dc = fc - pos[1]
                best_dir = (dr, dc)
        if best_dir is None:
            return None
        dr, dc = best_dir
        best_action = None
        best_score = float("inf")
        for action, (mr, mc) in self.movement_mapping.items():
            score = abs(mr - dr) + abs(mc - dc)
            if score < best_score:
                best_score = score
                best_action = action
        return best_action

    def get_explore_ratio(self):
        total = len(self.floor_cells)
        if total == 0:
            return 0.0
        return len(self.visited) / total


# Object Tracker
class TrackedObject:
    def __init__(self, obj_id, cells, color, frame_num):
        self.obj_id = obj_id
        self.color = color
        self.cells = set(cells)
        self.size = len(cells)
        self.centroid_r = sum(r for r, c in cells) / max(len(cells), 1)
        self.centroid_c = sum(c for r, c in cells) / max(len(cells), 1)
        self.first_seen = frame_num
        self.last_seen = frame_num
        self.changed_recently = False
        self.interaction_attempts = 0

    def update_position(self, cells, frame_num):
        old_cr, old_cc = self.centroid_r, self.centroid_c
        self.cells = set(cells)
        self.size = len(cells)
        self.centroid_r = sum(r for r, c in cells) / max(len(cells), 1)
        self.centroid_c = sum(c for r, c in cells) / max(len(cells), 1)
        self.last_seen = frame_num
        dist = math.sqrt((self.centroid_r - old_cr)**2 + (self.centroid_c - old_cc)**2)
        self.changed_recently = dist > 0.5
        return self.changed_recently


class ObjectTracker:
    def __init__(self):
        self.objects = {}
        self.next_obj_id = 1
        self.disappeared_objects = set()
        self.new_objects = set()

    def reset_for_level(self):
        self.objects.clear()
        self.next_obj_id = 1
        self.disappeared_objects.clear()
        self.new_objects.clear()

    def _find_connected_components(self, grid):
        if grid is None:
            return []
        h, w = grid.shape
        visited = np.zeros((h, w), dtype=bool)
        components = []
        for r in range(h):
            for c in range(w):
                if grid[r, c] != Config.BACKGROUND_COLOR and not visited[r, c]:
                    component = []
                    queue = deque([(r, c)])
                    visited[r, c] = True
                    color = grid[r, c]
                    while queue:
                        cr, cc = queue.popleft()
                        component.append((cr, cc))
                        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                            nr, nc = cr + dr, cc + dc
                            if (0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color):
                                visited[nr, nc] = True
                                queue.append((nr, nc))
                    components.append(component)
        return components

    def update(self, grid, frame_num):
        if grid is None:
            return
        self.new_objects.clear()
        self.disappeared_objects.clear()
        components = self._find_connected_components(grid)
        matched_obj_ids = set()
        matched_comp_ids = set()
        for obj_id, obj in self.objects.items():
            best_comp_idx = -1
            best_overlap = 0
            for i, comp in enumerate(components):
                if i in matched_comp_ids:
                    continue
                overlap = len(obj.cells & set(comp))
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_comp_idx = i
            if best_comp_idx >= 0 and best_overlap >= max(1, obj.size * 0.3):
                obj.update_position(components[best_comp_idx], frame_num)
                matched_obj_ids.add(obj_id)
                matched_comp_ids.add(best_comp_idx)
        for obj_id in self.objects:
            if obj_id not in matched_obj_ids:
                self.disappeared_objects.add(obj_id)
        for i, comp in enumerate(components):
            if i not in matched_comp_ids and len(comp) >= Config.MIN_OBJECT_SIZE:
                r0, c0 = comp[0]
                color = int(grid[r0, c0])
                obj = TrackedObject(self.next_obj_id, comp, color, frame_num)
                self.objects[self.next_obj_id] = obj
                self.new_objects.add(self.next_obj_id)
                self.next_obj_id += 1

    def get_click_candidates(self, player_pos=None, max_candidates=10, tried_clicks=None):
        if tried_clicks is None:
            tried_clicks = set()
        candidates = []
        for obj_id, obj in self.objects.items():
            if obj.interaction_attempts > 3:
                continue
            cr, cc = int(obj.centroid_r), int(obj.centroid_c)
            if (cr, cc) in tried_clicks:
                continue
            score = 0.0
            if obj_id in self.new_objects:
                score += 50.0
            if obj.changed_recently:
                score += 30.0
            if player_pos is not None:
                dist = manhattan_distance(player_pos[0], player_pos[1], cr, cc)
                score += max(0, 20 - dist)
            if obj.size <= 4:
                score += 15.0
            elif obj.size <= 9:
                score += 10.0
            score += max(0, 10 - obj.interaction_attempts * 5)
            if score > 0:
                candidates.append((cr, cc, score))
        candidates.sort(key=lambda x: x[2], reverse=True)
        return candidates[:max_candidates]

    def get_object_summary(self):
        return f"Objects: {len(self.objects)} tracked, {len(self.new_objects)} new, {len(self.disappeared_objects)} disappeared"


# Stuck Detector
class StuckDetector:
    def __init__(self):
        self.state_visit_counts = Counter()
        self.no_progress_count = 0
        self.loop_detected = False
        self.stuck_count = 0
        self.recovery_mode = False
        self.recovery_action_idx = 0
        self.last_progress_hash = None
        self.recovery_actions = ["ACTION5", "ACTION7", "ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5"]

    def reset_for_level(self):
        self.state_visit_counts.clear()
        self.no_progress_count = 0
        self.loop_detected = False
        self.stuck_count = 0
        self.recovery_mode = False
        self.recovery_action_idx = 0
        self.last_progress_hash = None

    def update(self, state_hash, levels_completed, prev_levels):
        status = {"loop_detected": False, "no_progress": False, "should_recover": False, "progress_made": levels_completed > prev_levels}
        self.state_visit_counts[state_hash] += 1
        if self.state_visit_counts[state_hash] >= Config.LOOP_THRESHOLD:
            status["loop_detected"] = True
            self.loop_detected = True
            logger.info(f"Loop detected: state visited {self.state_visit_counts[state_hash]} times")
        if state_hash == self.last_progress_hash:
            self.no_progress_count += 1
        else:
            self.no_progress_count = 0
            self.last_progress_hash = state_hash
        if self.no_progress_count >= Config.NO_PROGRESS_THRESHOLD:
            status["no_progress"] = True
        if status["loop_detected"] or status["no_progress"]:
            self.stuck_count += 1
            if self.stuck_count >= 2:
                status["should_recover"] = True
                if not self.recovery_mode:
                    self.recovery_mode = True
                    self.recovery_action_idx = 0
                    logger.info("Starting stuck recovery mode")
        if status["progress_made"]:
            self.recovery_mode = False
            self.stuck_count = 0
        return status

    def get_recovery_action(self, available_actions):
        if not self.recovery_mode:
            return None
        if self.recovery_action_idx < len(self.recovery_actions):
            action = self.recovery_actions[self.recovery_action_idx]
            self.recovery_action_idx += 1
            return action
        if self.recovery_action_idx < len(self.recovery_actions) + 3:
            self.recovery_action_idx += 1
            return "ACTION6"
        self.recovery_mode = False
        return None


# Level Progression Manager
class LevelProgressionManager:
    def __init__(self):
        self.current_level = 0
        self.levels_completed = 0
        self.win_levels = 1
        self.cross_level_knowledge = {"movement_verified": False, "interact_useful": False, "click_useful": False}
        self.actions_since_level_start = 0
        self.level_action_efficiency = []

    def update(self, levels_completed, win_levels):
        info = {"new_level": False, "level_completed": False, "game_won": False}
        if levels_completed > self.levels_completed:
            info["level_completed"] = True
            info["new_level"] = True
            self.level_action_efficiency.append(self.actions_since_level_start)
            self.actions_since_level_start = 0
        self.levels_completed = levels_completed
        self.current_level = levels_completed
        self.win_levels = win_levels
        if levels_completed >= win_levels and win_levels > 0:
            info["game_won"] = True
        return info

    def record_knowledge(self, key, value=True):
        self.cross_level_knowledge[key] = value

    def increment_actions(self):
        self.actions_since_level_start += 1

    def get_summary(self):
        return f"Level {self.current_level}/{self.win_levels}, {len(self.level_action_efficiency)} levels completed"


# Main SmartAgent (core logic)
class SmartAgent:
    def __init__(self):
        self.game_id = ""
        self.frames = []
        self.action_counter = 0
        self.hypothesis_tracker = HypothesisTracker()
        self.spatial_memory = SpatialMemory()
        self.object_tracker = ObjectTracker()
        self.stuck_detector = StuckDetector()
        self.level_manager = LevelProgressionManager()
        self.prev_frame_hash = ""
        self.prev_grid = None
        self.prev_action_name = ""
        self.prev_levels_completed = 0
        self.tried_clicks = set()
        self.click_count = 0
        self.click_effect_count = 0
        self._available_actions_cache = []
        self._started = False
        self._initial_reset_done = False
        self._explore_phase = 0
        self._direction_test_idx = 0
        self._direction_test_order = ["ACTION1", "ACTION2", "ACTION3", "ACTION4"]
        self._interact_tested = False
        self._undo_tested = False
        self._phase_actions_count = 0
        self.action_history = []

    def _get_grid(self, frame):
        try:
            if hasattr(frame, "frame") and frame.frame:
                return grid_to_np(frame.frame)
        except Exception:
            pass
        return None

    def _make_action(self, action_name, x=None, y=None, reasoning=""):
        from arcengine import GameAction
        action_map = {"RESET": GameAction.RESET, "ACTION1": GameAction.ACTION1, "ACTION2": GameAction.ACTION2, "ACTION3": GameAction.ACTION3, "ACTION4": GameAction.ACTION4, "ACTION5": GameAction.ACTION5, "ACTION6": GameAction.ACTION6, "ACTION7": GameAction.ACTION7}
        action = action_map.get(action_name, GameAction.ACTION5)
        if action_name == "ACTION6" and x is not None and y is not None:
            action.set_data({"x": int(x), "y": int(y)})
        if reasoning:
            action.reasoning = reasoning
        return action

    def is_done(self, frames, latest_frame):
        try:
            from arcengine import GameState
            if latest_frame.state == GameState.WIN:
                logger.info(f"[{self.game_id}] Game WON!")
                return True
        except Exception:
            pass
        if self.action_counter >= Config.MAX_ACTIONS:
            return True
        return False

    def choose_action(self, frames, latest_frame):
        try:
            return self._choose_impl(frames, latest_frame)
        except Exception as e:
            logger.error(f"Error in choose_action: {e}")
            try:
                from arcengine import GameState
                if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                    return self._make_action("RESET", reasoning="fallback reset")
            except Exception:
                pass
            return self._make_action("ACTION1", reasoning="error fallback")

    def _choose_impl(self, frames, latest_frame):
        self.frames = frames
        current_grid = self._get_grid(latest_frame)
        current_hash = grid_hash(latest_frame.frame) if hasattr(latest_frame, "frame") else ""
        levels = getattr(latest_frame, "levels_completed", 0)
        win_levels = getattr(latest_frame, "win_levels", 1)
        available = getattr(latest_frame, "available_actions", []) or []
        if available:
            self._available_actions_cache = available
        level_info = self.level_manager.update(levels, win_levels)
        if level_info["new_level"]:
            self._on_new_level(latest_frame)
        if level_info["game_won"]:
            return self._make_action("RESET", reasoning="game won")
        try:
            from arcengine import GameState
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                self._initial_reset_done = False
                self._on_new_level(latest_frame)
                return self._make_action("RESET", reasoning=f"state={latest_frame.state}")
        except Exception:
            pass
        if not self._initial_reset_done:
            self._initial_reset_done = True
            self.hypothesis_tracker.add_initial_hypotheses(self._available_actions_cache or available)
            return self._make_action("ACTION1", reasoning="first exploration")
        self._update_subsystems(latest_frame, current_grid, current_hash)
        stuck_status = self.stuck_detector.update(current_hash, levels, self.prev_levels_completed)
        self.prev_levels_completed = levels
        if stuck_status["should_recover"] or self.stuck_detector.recovery_mode:
            action = self._handle_stuck_recovery(available)
            if action is not None:
                self._record_action(action, latest_frame)
                return action
        action = self._select_action(latest_frame, current_grid, available, current_hash)
        self._record_action(action, latest_frame)
        return action

    def _on_new_level(self, frame):
        logger.info(f"[{self.game_id}] New level, resetting subsystems")
        self.spatial_memory.reset_for_level()
        self.object_tracker.reset_for_level()
        self.stuck_detector.reset_for_level()
        self.tried_clicks.clear()
        self._explore_phase = 0
        self._direction_test_idx = 0
        self._interact_tested = False
        self._undo_tested = False
        self._phase_actions_count = 0
        self.prev_frame_hash = ""
        self.prev_grid = None
        self.prev_action_name = ""
        if self.spatial_memory.movement_mapping:
            self.level_manager.record_knowledge("movement_verified", True)

    def _update_subsystems(self, frame, grid, current_hash):
        if grid is None:
            return
        self.spatial_memory.update(grid, self.prev_grid, self.prev_action_name)
        if len(self.frames) % 2 == 0:
            self.object_tracker.update(grid, len(self.frames))
        if self.spatial_memory.player_pos:
            r, c = self.spatial_memory.player_pos
            self.spatial_memory.mark_area_visited(r, c)
        if self.prev_frame_hash and self.prev_action_name:
            prev_levels = self.prev_levels_completed
            curr_levels = getattr(frame, "levels_completed", 0)
            self.hypothesis_tracker.record_action_result(self.prev_action_name, self.prev_frame_hash, current_hash, prev_levels, curr_levels, getattr(frame, "state", None))
        self.prev_frame_hash = current_hash
        self.prev_grid = grid.copy() if grid is not None else None

    def _handle_stuck_recovery(self, available):
        recovery_action_name = self.stuck_detector.get_recovery_action(available)
        if recovery_action_name is None:
            simple = [a.name for a in available if a.is_simple() and a.name != "RESET"]
            if simple:
                return self._make_action(random.choice(simple), reasoning="random recovery")
            candidates = self.object_tracker.get_click_candidates(self.spatial_memory.player_pos, tried_clicks=self.tried_clicks)
            if candidates:
                r, c, _ = candidates[0]
                return self._make_action("ACTION6", x=c, y=r, reasoning="stuck click")
            return self._make_action("ACTION6", x=random.randint(0, 30), y=random.randint(0, 30), reasoning="random click recovery")
        if recovery_action_name == "ACTION6":
            candidates = self.object_tracker.get_click_candidates(self.spatial_memory.player_pos, tried_clicks=self.tried_clicks)
            if candidates:
                r, c, _ = random.choice(candidates)
                return self._make_action("ACTION6", x=c, y=r, reasoning="recovery click")
            return self._make_action("ACTION6", x=random.randint(0, 30), y=random.randint(0, 30), reasoning="random recovery click")
        return self._make_action(recovery_action_name, reasoning=f"stuck recovery: {recovery_action_name}")

    def _select_action(self, frame, grid, available, current_hash):
        self._phase_actions_count += 1
        self.level_manager.increment_actions()
        if random.random() < Config.RANDOM_EXPLORATION_PROB:
            simple = [a.name for a in available if a.is_simple() and a.name != "RESET"]
            if simple:
                return self._make_action(random.choice(simple), reasoning="random exploration")
        if self._explore_phase == 0:
            return self._phase_initial_exploration(available)
        if self._explore_phase == 1:
            return self._phase_directed_exploration(frame, grid, available)
        if self._explore_phase == 2:
            return self._phase_exploitation(frame, grid, available)
        return self._make_action("ACTION1", reasoning="default fallback")

    def _phase_initial_exploration(self, available):
        if self._direction_test_idx < len(self._direction_test_order):
            action_name = self._direction_test_order[self._direction_test_idx]
            self._direction_test_idx += 1
            return self._make_action(action_name, reasoning=f"initial: test {action_name}")
        if not self._interact_tested:
            self._interact_tested = True
            return self._make_action("ACTION5", reasoning="initial: test interact")
        if not self._undo_tested:
            self._undo_tested = True
            return self._make_action("ACTION7", reasoning="initial: test undo")
        candidates = self.object_tracker.get_click_candidates(self.spatial_memory.player_pos, tried_clicks=self.tried_clicks)
        if candidates:
            r, c, _ = candidates[0]
            self.tried_clicks.add((r, c))
            return self._make_action("ACTION6", x=c, y=r, reasoning=f"initial: click ({c},{r})")
        self._explore_phase = 1
        self._phase_actions_count = 0
        return self._make_action("ACTION1", reasoning="transition to directed exploration")

    def _phase_directed_exploration(self, frame, grid, available):
        if self._phase_actions_count % 10 == 0:
            return self._make_action("ACTION5", reasoning="periodic interact")
        if self._phase_actions_count % 15 == 0:
            return self._make_action("ACTION7", reasoning="periodic undo")
        if self._phase_actions_count % 8 == 0:
            click_action = self._try_smart_click(available)
            if click_action is not None:
                return click_action
        frontier_dir = self.spatial_memory.get_unexplored_direction()
        if frontier_dir is not None:
            return self._make_action(frontier_dir, reasoning=f"move to frontier: {frontier_dir}")
        untested = self.hypothesis_tracker.get_best_untested_action(available)
        if untested is not None:
            return self._make_action(untested, reasoning=f"test hypothesis: {untested}")
        if self.spatial_memory.movement_mapping:
            action = self._systematic_explore(available)
            if action is not None:
                return action
        remaining = [a for a in self._direction_test_order if a not in self.spatial_memory.movement_mapping]
        if remaining:
            return self._make_action(remaining[0], reasoning=f"re-test: {remaining[0]}")
        simple = [a.name for a in available if a.is_simple() and a.name in ("ACTION1", "ACTION2", "ACTION3", "ACTION4")]
        if simple:
            return self._make_action(random.choice(simple), reasoning="random movement")
        click_action = self._try_smart_click(available)
        if click_action is not None:
            return click_action
        return self._make_action("ACTION1", reasoning="directed fallback")

    def _phase_exploitation(self, frame, grid, available):
        productive = self._find_productive_actions()
        if productive:
            action_name, score = productive[0]
            return self._make_action(action_name, reasoning=f"exploit: {action_name} (score={score:.1f})")
        return self._phase_directed_exploration(frame, grid, available)

    def _systematic_explore(self, available):
        if not self.spatial_memory.movement_mapping:
            return None
        pos = self.spatial_memory.player_pos
        if pos is None:
            return None
        best_action = None
        best_unvisited_ratio = -1
        for action_name, (dr, dc) in self.spatial_memory.movement_mapping.items():
            nr, nc = pos[0] + dr, pos[1] + dc
            if (nr, nc) in self.spatial_memory.wall_cells:
                continue
            unvisited = 0
            total = 0
            for ddr in range(-2, 3):
                for ddc in range(-2, 3):
                    r, c = nr + ddr, nc + ddc
                    if (0 <= r < self.spatial_memory.grid_height and 0 <= c < self.spatial_memory.grid_width):
                        total += 1
                        if (r, c) not in self.spatial_memory.visited:
                            unvisited += 1
            ratio = unvisited / max(total, 1)
            if ratio > best_unvisited_ratio:
                best_unvisited_ratio = ratio
                best_action = action_name
        if best_action:
            return self._make_action(best_action, reasoning=f"systematic ({best_unvisited_ratio:.0%} unvisited)")
        return None

    def _try_smart_click(self, available):
        has_click = any((hasattr(a, "name") and a.name == "ACTION6") or (hasattr(a, "is_complex") and a.is_complex()) for a in available)
        if not has_click:
            return None
        candidates = self.object_tracker.get_click_candidates(self.spatial_memory.player_pos, max_candidates=Config.CLICK_CANDIDATE_LIMIT, tried_clicks=self.tried_clicks)
        if not candidates:
            frontiers = self.spatial_memory.get_frontier_cells()
            if frontiers:
                r, c = random.choice(frontiers)
                self.tried_clicks.add((r, c))
                return self._make_action("ACTION6", x=c, y=r, reasoning=f"click frontier ({c},{r})")
            return None
        r, c, score = candidates[0]
        self.tried_clicks.add((r, c))
        self.click_count += 1
        for obj_id, obj in self.object_tracker.objects.items():
            if manhattan_distance(int(obj.centroid_r), int(obj.centroid_c), r, c) <= 1:
                obj.interaction_attempts += 1
        return self._make_action("ACTION6", x=c, y=r, reasoning=f"smart click ({c},{r}) prio={score:.1f}")

    def _find_productive_actions(self):
        action_scores = defaultdict(float)
        for entry in self.action_history:
            name = entry.get("action_name", "")
            if not name or name == "RESET":
                continue
            score = 0
            if entry.get("changed_state"): score += 1.0
            if entry.get("level_up"): score += 10.0
            if entry.get("new_object"): score += 3.0
            if not entry.get("changed_state"): score -= 0.5
            action_scores[name] += score
        scored = [(name, score) for name, score in action_scores.items() if score > 0]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def _record_action(self, action, frame):
        action_name = ""
        try:
            action_name = action.name
        except Exception:
            action_name = str(action)
        changed = self.prev_frame_hash != (grid_hash(frame.frame) if hasattr(frame, "frame") else "")
        new_objects = len(self.object_tracker.new_objects) > 0
        level_up = getattr(frame, "levels_completed", 0) > self.prev_levels_completed
        self.action_history.append({"step": self.action_counter, "action_name": action_name, "changed_state": changed, "new_object": new_objects, "level_up": level_up})
        self.prev_action_name = action_name
        if action_name == "ACTION6" and changed:
            self.click_effect_count += 1
        if self._explore_phase == 1 and self._phase_actions_count > 20 and self.spatial_memory.get_explore_ratio() > 0.5:
            self._explore_phase = 2
            self._phase_actions_count = 0

    def get_diagnostics(self):
        return {"action_counter": self.action_counter, "explore_phase": self._explore_phase, "hypotheses": self.hypothesis_tracker.get_summary(), "spatial": f"visited={len(self.spatial_memory.visited)}, walls={len(self.spatial_memory.wall_cells)}, ratio={self.spatial_memory.get_explore_ratio():.1%}", "objects": self.object_tracker.get_object_summary(), "stuck": f"loop={self.stuck_detector.loop_detected}, recovery={self.stuck_detector.recovery_mode}", "levels": self.level_manager.get_summary(), "clicks": f"{self.click_count} tried, {self.click_effect_count} effective", "movement": self.spatial_memory.movement_mapping}


# Agent that inherits from the official Agent base class
class StandaloneSmartAgent:
    MAX_ACTIONS = Config.MAX_ACTIONS

    def __init__(self, *args, **kwargs):
        from agents.agent import Agent
        Agent.__init__(self, *args, **kwargs)
        self._smart = SmartAgent()
        self._smart.game_id = self.game_id

    def is_done(self, frames, latest_frame):
        try:
            from arcengine import GameState
            if latest_frame.state == GameState.WIN:
                logger.info(f"[{self.game_id}] Game WON!")
                return True
        except Exception:
            pass
        if self.action_counter >= self.MAX_ACTIONS:
            logger.info(f"[{self.game_id}] Reached MAX_ACTIONS ({self.MAX_ACTIONS})")
            return True
        return False

    def choose_action(self, frames, latest_frame):
        self._smart.frames = frames
        self._smart.action_counter = self.action_counter
        try:
            action = self._smart.choose_action(frames, latest_frame)
        except Exception as e:
            logger.error(f"[{self.game_id}] Error: {e}")
            from arcengine import GameState, GameAction
            try:
                if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                    return GameAction.RESET
            except Exception:
                pass
            action = GameAction.ACTION1
        self.action_counter += 1
        if self.action_counter % 20 == 0:
            diag = self._smart.get_diagnostics()
            logger.info(f"[{self.game_id}] Diag: {diag}")
        return action

    @property
    def name(self):
        try:
            base = super().name
        except Exception:
            base = self.game_id
        return f"{base}.smart_v2.{self.MAX_ACTIONS}"
print("SmartAgent V2 loaded successfully")
print(f"MAX_ACTIONS = {Config.MAX_ACTIONS}")
print(f"LOOP_THRESHOLD = {Config.LOOP_THRESHOLD}")
print(f"NO_PROGRESS_THRESHOLD = {Config.NO_PROGRESS_THRESHOLD}")
print(f"Components: HypothesisTracker, SpatialMemory, ObjectTracker, StuckDetector, LevelProgressionManager")

## Competition Execution

The cell below handles both **competition mode** (Kaggle rerun) and **development mode** (local testing). In competition mode, it:
1. Waits for the gateway to be ready
2. Copies the agent code into the ARC-AGI-3-Agents framework
3. Registers the agent in `__init__.py`
4. Configures environment variables
5. Runs the agent on all games via the Swarm orchestrator

In [ ]:
import os, subprocess, time

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('=== COMPETITION MODE ===')
    
    # Wait for gateway to be ready
    for i in range(60):
        r = subprocess.run(['curl', '-sf', 'http://gateway:8001/api/games'],
                          capture_output=True, timeout=10)
        if r.returncode == 0:
            print('Gateway ready')
            break
        time.sleep(5)
    
    # Setup agent framework
    WORK = '/kaggle/working/ARC-AGI-3-Agents'
    SRC = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
    
    if os.path.exists(SRC):
        subprocess.run(['cp', '-r', SRC, WORK], check=True)
        subprocess.run(['cp', '/kaggle/working/my_agent.py',
                       f'{WORK}/agents/templates/smart_agent_v2.py'], check=True)
        
        # Register agent in __init__.py
        init_code = '''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.smart_agent_v2 import StandaloneSmartAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "smartv2": StandaloneSmartAgent,
}
'''
        with open(f'{WORK}/agents/__init__.py', 'w') as f:
            f.write(init_code)
        
        # Configure environment
        env_content = 'SCHEME=http\\nHOST=gateway\\nPORT=8001\\nARC_API_KEY=***\\nARC_BASE_URL=http://gateway:8001/\\nOPERATION_MODE=online\\nRECORDINGS_DIR=/kaggle/working/server_recording\\n'
        with open(f'{WORK}/.env', 'w') as f:
            f.write(env_content)
        
        os.chdir(WORK)
        os.environ['MPLBACKEND'] = 'agg'
        
        # Run the agent on all games
        print('Starting SmartAgent V2 on all games...')
        result = subprocess.run(
            ['python', 'main.py', '--agent', 'smartv2'],
            capture_output=True, text=True, timeout=3600
        )
        
        print(f'Exit code: {result.returncode}')
        if result.stdout:
            print('STDOUT:', result.stdout[-2000:])
        if result.stderr:
            print('STDERR:', result.stderr[-1000:])
    else:
        print(f"Source not found at {SRC}")
        for d in os.listdir('/kaggle/input/'):
            print(f"  {d}")
else:
    print('=== DEVELOPMENT MODE (not competition rerun) ===')
    print('SmartAgent V2 is prepared.')
    print('Submit to competition to run on all games.')
    print()
    print('Agent components loaded:')
    print('  - HypothesisTracker: Forms/test conjectures about game mechanics')
    print('  - SpatialMemory: 2D explored map with player tracking')
    print('  - ObjectTracker: Connected component detection + tracking')
    print('  - StuckDetector: Loop detection + multi-strategy recovery')
    print('  - LevelProgressionManager: Cross-level knowledge transfer')
    print('  - 3-Phase exploration: Initial -> Directed -> Exploitation')